# LIKAS — Sample Prompts for the Fine-Tuned Model

A hands-on demo of [`jpcurada/likas-gemma4-e2b-gguf`](https://huggingface.co/jpcurada/likas-gemma4-e2b-gguf) — the on-device disaster-response model that powers the [LIKAS](https://github.com/JpCurada/likas) app.

This notebook loads the **same Q4_K_M GGUF the phone runs** (via `llama-cpp-python`, which is the same `llama.cpp` engine `llama.rn` uses on-device) and walks through prompts that exercise every capability the model was fine-tuned for:

| Section | Capability |
|---|---|
| III | `get_protocol` — quote NDRRMC / PAGASA / PHIVOLCS verbatim |
| IV | `route_to_nearest_evacuation` — evacuation routing |
| V | `find_nearby` — POI search (hospital / school / gym …) |
| VI | `get_user_profile` — personalized answers |
| VII | Multilingual — English / Filipino / Taglish (Rule 8) |
| VIII | Refusal of off-topic queries (Rule 7) |
| IX | The full multi-turn tool-dispatch loop end-to-end |

> ⚠️ This is a **task-specific fine-tune**, not a chatbot. It emits exactly one JSON envelope per turn — either `{"action":"tool",...}` or `{"action":"speak",...}`. The cells below reproduce the **exact production system prompt** from `Likas/src/services/aiAssistantService.ts` so the model behaves the way it does in the app. Run top-to-bottom on Kaggle/Colab (CPU is fine) or locally with `llama-cpp-python` installed.

## I. Setup

Download the published GGUF and load it. ~1.8 GB; the `hf_hub_download` call caches so a re-run is instant.

In [ ]:
%%capture
!pip install -q -U "huggingface_hub>=1.5.0,<2.0" llama-cpp-python

In [ ]:
from huggingface_hub import hf_hub_download
from llama_cpp import Llama, LlamaGrammar
import json

GGUF_REPO = "jpcurada/likas-gemma4-e2b-gguf"
GGUF_FILE = "likas-q4_k_m.gguf"

model_path = hf_hub_download(repo_id=GGUF_REPO, filename=GGUF_FILE)
print("GGUF cached at:", model_path)

# n_ctx=4096 matches the training max_seq_length and the app's context size.
# n_gpu_layers=-1 offloads all layers if a GPU is present; harmless on CPU.
llm = Llama(model_path=model_path, n_ctx=4096, n_gpu_layers=-1, verbose=False)
print("Model loaded.")

## II. Reproduce the production runtime

Three pieces, copied 1:1 from the app so the model sees the **same distribution as at inference time on-device**:

1. **The tool registry** — same four tools, same descriptions/params as `Likas/src/services/aiTools.ts`.
2. **`build_system_prompt()`** — a faithful Python port of `buildSystemPrompt()` in `aiAssistantService.ts`, including the 9 CRITICAL RULES and the profile summary.
3. **A GBNF grammar** — the app forces the model to emit a valid JSON envelope even at temperature 1.0 via `aiGrammar.ts`. We apply an equivalent grammar here so output is always parseable.

A tiny `dispatch()` helper runs one user turn (no tools resolved); `dispatch_loop()` in Section IX runs the full multi-turn loop with mocked tool results.

In [ ]:
# --- Tool registry (mirrors Likas/src/services/aiTools.ts) ---
TOOLS = [
    {
        "name": "route_to_nearest_evacuation",
        "description": "Rank the top 3 nearest evacuation centers for the user, considering distance, PWD/elderly access, and pet-friendliness. Use this when the user asks where to evacuate, where to go, or about the nearest shelter.",
        "properties": {"profile_aware": {"type": "boolean", "description": "When true, factor in the user profile (PWD, elderly, pets). Defaults to true."}},
    },
    {
        "name": "get_protocol",
        "description": "Look up the official NDRRMC/PHIVOLCS/PAGASA protocol for a disaster type and phase. Use this for ANY safety-critical question. Quote the returned text verbatim — do not paraphrase.",
        "properties": {
            "disaster": {"type": "string", "enum": ["earthquake", "typhoon", "volcano"], "description": "The disaster type to look up."},
            "phase": {"type": "string", "enum": ["before", "during", "after"], "description": "The phase of the disaster: before (preparation), during (active event), after (recovery)."},
        },
    },
    {
        "name": "find_nearby",
        "description": "Find the 3 nearest places of interest in a category (hospital, evacuation center, school, gymnasium, multi_purpose_hall, covered_court).",
        "properties": {"category": {"type": "string", "enum": ["hospital", "evacuation_center", "gymnasium", "school", "multi_purpose_hall", "covered_court"], "description": "The POI category to search."}},
    },
    {
        "name": "get_user_profile",
        "description": "Return a detailed view of the user profile beyond the summary in the system prompt: medical conditions, meeting points, emergency contacts, exact address, pet details. Use when the user asks about themselves or when those details would meaningfully change your advice.",
        "properties": {},
    },
]

In [ ]:
# --- Python port of buildSystemPrompt() in aiAssistantService.ts ---
def build_system_prompt(profile_summary: str, active_context: str) -> str:
    tool_list = "\n".join(
        f"- {t['name']}({json.dumps(t['properties'])}): {t['description']}"
        for t in TOOLS
    )
    return f"""You are LIKAS, an offline disaster companion for the Philippines.

CRITICAL RULES — VIOLATING THESE PUTS LIVES AT RISK:
1. For ANY safety-critical question, you MUST call get_protocol first and quote its returned text verbatim. Never invent or paraphrase safety steps.
2. For ANY question about evacuation, where to go, or shelters, you MUST call route_to_nearest_evacuation.
3. If asked to find a hospital, school, gym, or other facility, call find_nearby.
4. If the user asks about their own profile (medical conditions, meeting points, emergency contacts) call get_user_profile.
5. PERSONALIZE every reply using the USER PROFILE below. Mention the user's name when natural. If they have asthma, prioritize masks. If they have an infant/elderly/pwd companion, factor that into evacuation timing. If they have pets, address pet logistics. Reference their primary meeting point when discussing family reunification.
6. If unsure, respond with: "I can't verify that protocol — contact NDRRMC at 911."
7. Refuse off-topic questions (entertainment, opinions, general knowledge) and redirect to disaster topics.
8. Respond in the same language the user used (English, Filipino, or Taglish). Keep replies concise.
9. NEVER send SMS, place calls, or take real-world actions — those are user-controlled only.

OUTPUT FORMAT — STRICT JSON, NO PROSE OUTSIDE JSON:
- To call a tool: {{"action":"tool","name":"<tool_name>","args":{{...}}}}
- To answer the user: {{"action":"speak","text":"<your reply>"}}
- Output exactly ONE JSON object per turn. After a tool result is returned, decide again.

AVAILABLE TOOLS:
{tool_list}

ACTIVE DISASTER CONTEXT: {active_context}
USER PROFILE: {profile_summary}"""


# A representative profile (same shape buildSystemPrompt emits). Edit freely
# to see how Rule 5 personalization changes the answers.
DEMO_PROFILE = (
    "name=Maria; ageGroup=adult; "
    "companions: infants=1, children=0, elderly=1, pwd=0; "
    "pets: 1 dogs (medium); medicalConditions: asthma; "
    "address=12 Mabini St, Barangay 659, Manila; "
    "primaryMeetingPoint=Rizal Park flagpole; "
    "emergencyContacts: Juan Dela Cruz (brother)"
)
print(build_system_prompt(DEMO_PROFILE, "earthquake")[:600], "...")

In [ ]:
# --- GBNF grammar: force a valid JSON envelope (equivalent to aiGrammar.ts) ---
# speak | one of the four tool calls. Tool names + enums are baked in as
# literals exactly like buildGrammar() does, so output is always parseable.
GBNF = r'''
root        ::= speak | tool_route | tool_protocol | tool_nearby | tool_profile
ws          ::= [ \t\n]*
string      ::= "\"" ( [^"\\] | "\\" . )* "\""
boolean     ::= "true" | "false"

speak       ::= "{" ws "\"action\"" ws ":" ws "\"speak\"" ws "," ws "\"text\"" ws ":" ws string ws "}"

tool_route  ::= "{" ws "\"action\"" ws ":" ws "\"tool\"" ws "," ws "\"name\"" ws ":" ws "\"route_to_nearest_evacuation\"" ws "," ws "\"args\"" ws ":" ws args_route ws "}"
args_route  ::= "{}" | "{" ws "\"profile_aware\"" ws ":" ws boolean ws "}"

tool_protocol ::= "{" ws "\"action\"" ws ":" ws "\"tool\"" ws "," ws "\"name\"" ws ":" ws "\"get_protocol\"" ws "," ws "\"args\"" ws ":" ws args_protocol ws "}"
args_protocol ::= "{" ws "\"disaster\"" ws ":" ws disaster ws "," ws "\"phase\"" ws ":" ws phase ws "}"
disaster    ::= "\"earthquake\"" | "\"typhoon\"" | "\"volcano\""
phase       ::= "\"before\"" | "\"during\"" | "\"after\""

tool_nearby ::= "{" ws "\"action\"" ws ":" ws "\"tool\"" ws "," ws "\"name\"" ws ":" ws "\"find_nearby\"" ws "," ws "\"args\"" ws ":" ws args_nearby ws "}"
args_nearby ::= "{" ws "\"category\"" ws ":" ws category ws "}"
category    ::= "\"hospital\"" | "\"evacuation_center\"" | "\"gymnasium\"" | "\"school\"" | "\"multi_purpose_hall\"" | "\"covered_court\""

tool_profile ::= "{" ws "\"action\"" ws ":" ws "\"tool\"" ws "," ws "\"name\"" ws ":" ws "\"get_user_profile\"" ws "," ws "\"args\"" ws ":" ws "{}" ws "}"
'''
GRAMMAR = LlamaGrammar.from_string(GBNF)
print("Grammar compiled.")

In [ ]:
def dispatch(user_message, context="earthquake", profile=DEMO_PROFILE,
             history=None, show_prompt=False):
    """Run ONE model turn and return the parsed JSON envelope.

    `history` is a list of prior {role, content} messages (used by the loop in
    Section IX to feed tool results back). Sampling matches the app: temp 0.4,
    top_p 0.85, grammar-constrained so the output always parses.
    """
    messages = [{"role": "system", "content": build_system_prompt(profile, context)}]
    if history:
        messages += history
    messages.append({"role": "user", "content": user_message})
    if show_prompt:
        print(messages[-1])
    out = llm.create_chat_completion(
        messages=messages,
        grammar=GRAMMAR,
        temperature=0.4, top_p=0.85, top_k=40, repeat_penalty=1.1,
        max_tokens=512, stop=["<end_of_turn>"],
    )
    raw = out["choices"][0]["message"]["content"].strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"_unparsed": raw}


def show(title, user_message, **kw):
    print("=" * 72)
    print(f"▶ {title}")
    print(f"  user: {user_message!r}")
    result = dispatch(user_message, **kw)
    print("  model:", json.dumps(result, ensure_ascii=False))
    print()
    return result

## III. Protocol Q&A (`get_protocol`)

Rule 1: any safety-critical question must trigger a `get_protocol` tool call first. Note the model picks the correct `disaster` + `phase` enum from the user's wording and the active context.

In [ ]:
show("Earthquake, active event", "May lindol! Ano gagawin ko ngayon?", context="earthquake")
show("Typhoon, before it hits",  "A typhoon is coming tomorrow, how do I prepare?", context="typhoon")
show("Volcano, after eruption",  "The ashfall stopped. What now?", context="volcano")
show("First aid — bleeding",     "My arm is bleeding badly, what do I do?", context="earthquake")

## IV. Evacuation routing (`route_to_nearest_evacuation`)

Rule 2: anything about *where to go* / shelters must call this tool. It accepts an optional `profile_aware` flag (default true) so PWD/elderly/pet constraints are honored.

In [ ]:
show("Direct ask (Tagalog)",   "Saan ako lilikas?", context="typhoon")
show("Direct ask (English)",   "Where is the nearest evacuation center?", context="earthquake")
show("With pet concern",       "May aso ako, saan kaming pwedeng pumunta na safe?", context="typhoon")

## V. POI search (`find_nearby`)

Rule 3: requests to find a facility map to `find_nearby` with the right `category` enum.

In [ ]:
show("Hospital",          "Where's the closest hospital?", context="earthquake")
show("School (Tagalog)",  "May malapit bang paaralan dito?", context="typhoon")
show("Covered court",     "Is there a covered court nearby we can shelter in?", context="typhoon")

## VI. Profile-aware answers (`get_user_profile`)

Rule 4: self-referential questions call `get_user_profile`. Compare these against the `DEMO_PROFILE` defined in Section II.

In [ ]:
show("Meds question",        "What medical conditions do I have on file?", context="earthquake")
show("Meeting point",        "Saan ulit yung meeting place namin ng pamilya?", context="typhoon")
show("Emergency contacts",   "Who are my emergency contacts?", context="earthquake")

## VII. Multilingual (Rule 8)

The model replies in the **user's language** — English, Filipino, or Taglish. These all ask the same thing in different languages; watch the `speak` text (after the tool round-trip) match the input language.

In [ ]:
show("English", "What should I do during an earthquake?", context="earthquake")
show("Filipino", "Ano ang dapat kong gawin kapag may lindol?", context="earthquake")
show("Taglish",  "Pano ba mag-prepare for earthquake? Anong mga steps?", context="earthquake")

## VIII. Refusal of off-topic queries (Rule 7)

Out of scope by design — the model politely refuses and redirects to disaster topics instead of answering. It should emit a `speak` turn (no tool call).

In [ ]:
show("Entertainment",     "Who won the NBA finals last year?", context="earthquake")
show("General knowledge", "Write me a poem about the moon.", context="typhoon")
show("Opinion",           "What's your favorite food?", context="volcano")

## IX. The full multi-turn tool-dispatch loop

In production the host runs the tool, injects the result, and the model decides again — capped at 3 iterations (see `MAX_TOOL_CALLS_PER_TURN` in `aiAssistantService.ts`). Here we **mock** the tool results with strings shaped exactly like `aiTools.ts` returns, so you can see a complete `tool → result → speak` exchange.

This is the realistic end-to-end behavior — a single bare turn (Sections III–VIII) usually stops at the tool call; the verbatim-quoting and personalization happen on the **second** turn after the result comes back.

In [ ]:
# Mocked tool results, 1:1 in shape with Likas/src/services/aiTools.ts.
MOCK_TOOL_RESULTS = {
    "get_protocol": (
        "PHIVOLCS and NDRRMC guidance: DROP, COVER, AND HOLD ON. Get under a "
        "sturdy table and hold its leg. Stay away from windows and heavy "
        "furniture. After shaking stops, check for injuries, avoid elevators, "
        "and watch for aftershocks."
    ),
    "route_to_nearest_evacuation": (
        "Top evacuation options:\n"
        "1. Manila High School — 0.8 km (~11 min walking) [best match]\n"
        "2. Rajah Soliman Park Covered Court — 1.3 km (~18 min walking)\n"
        "3. Baseco Sports Complex — 2.1 km (~29 min walking)\n\n"
        "Route to Manila High School: 0.83 km along walkable roads, "
        "~11 min walking."
    ),
    "find_nearby": (
        "Nearest hospital:\n"
        "1. Ospital ng Maynila — 1.2 km · Quirino Ave cor Roxas Blvd\n"
        "2. Manila Doctors Hospital — 1.9 km · 667 United Nations Ave\n"
        "3. Philippine General Hospital — 2.4 km · Taft Ave, Ermita"
    ),
    "get_user_profile": (
        "Name: Maria\nAge group: adult\n"
        "Companions: 1 infants, 0 children, 1 elderly, 0 PWD\n"
        "Pets: 1 dogs (medium)\nMedical conditions: asthma\n"
        "Address: 12 Mabini St, Barangay 659, Manila\n"
        "Primary meeting point: Rizal Park flagpole\n"
        "Emergency contacts: Juan Dela Cruz — brother: 0917xxxxxxx"
    ),
}


def dispatch_loop(user_message, context="earthquake", profile=DEMO_PROFILE,
                  max_turns=3):
    """Full loop: model -> (tool -> mocked result -> model)* -> speak."""
    print("=" * 72)
    print(f"▶ {user_message!r}  [context={context}]")
    history = []
    for turn in range(max_turns):
        result = dispatch(user_message if turn == 0 else "", context,
                          profile, history=history or None)
        if turn == 0:
            history.append({"role": "user", "content": user_message})
        print(f"  [turn {turn}] model:", json.dumps(result, ensure_ascii=False))
        if result.get("action") == "speak":
            print(f"\n  ✓ FINAL REPLY:\n  {result['text']}\n")
            return result
        if result.get("action") == "tool":
            name = result["name"]
            tool_result = MOCK_TOOL_RESULTS.get(name, "(no mock result)")
            print(f"  [turn {turn}] tool({name}) -> {tool_result[:70]}...")
            # Feed the call + result back, exactly like the app does.
            history.append({"role": "assistant", "content": json.dumps(result)})
            history.append({"role": "tool",
                             "content": json.dumps({"name": name, "result": tool_result})})
            continue
        print("  (unparsed / unexpected envelope)")
        return result
    print("  (hit tool-call limit without a speak turn)")
    return None

In [ ]:
# Watch the model quote the protocol VERBATIM on the second turn (Rule 1)
# and personalize for Maria's asthma / elderly companion / dog (Rule 5).
dispatch_loop("May lindol! Ano gagawin ko?", context="earthquake")
dispatch_loop("Saan kami pwedeng lumikas? May aso at lola ako.", context="typhoon")
dispatch_loop("What meds do I have, and what should I pack?", context="earthquake")

## X. Try your own

Edit `DEMO_PROFILE` in Section II (re-run that cell) to change personalization, then call `dispatch_loop(...)` with any prompt. Things worth probing:

- **Multi-tool**: "Lindol! Saan ako pupunta?" — protocol *then* evacuation in one conversation.
- **Clarification**: "help" / "it's bad" — the model asks what's wrong before acting.
- **Language stickiness**: ask in Taglish and confirm the final `speak.text` stays Taglish.
- **Refusal**: any off-topic query should be a single `speak` turn that redirects.

See the [model card](https://huggingface.co/jpcurada/likas-gemma4-e2b-gguf) for the full evaluation rubric and limitations (Metro Manila scope, authority-snapshot drift, JSON-envelope requirement).

In [ ]:
dispatch_loop("Lindol! Saan ako pupunta?", context="earthquake")